# From Random Samples to Option Prices
## Monte Carlo in Practice
Manupriya Gupta, IIT Delhi MathSoc, November 10, 2026.

Run cells in order. This notebook is self-contained and uses synthetic inputs only. It follows the European call example in the slides. No API key or market dataset is required.


### 0. Environment
Use Python 3.11 or newer. NumPy 2.3.5 reproduces the slide environment. The next cell installs that version into the notebook environment. It requires an internet connection on first setup. The simulation itself does not use the network.


In [ ]:
%pip install -q numpy==2.3.5

In [ ]:
import math
import numpy as np
print("NumPy", np.__version__)


### 1. Define the model
A European call pays `max(S_T - K, 0)` at maturity. We assume a non-dividend-paying stock with geometric Brownian motion, constant volatility and a constant continuously compounded rate. Risk-neutral pricing uses drift `r`, not a stock-return forecast.


In [ ]:
S0, K, r, sigma, T = 100., 100., .05, .20, 1.
N = 100_000
seed = 42
rng = np.random.default_rng(seed)
z = rng.standard_normal(N)


### 2. Sample the terminal stock and discount the payoff
For this terminal-only payoff, the exact GBM terminal law is enough. No time grid is needed.


In [ ]:
st = S0 * np.exp((r - .5 * sigma**2) * T + sigma * np.sqrt(T) * z)
payoff = np.maximum(st - K, 0.)
x = np.exp(-r * T) * payoff
print("First five terminal prices:", st[:5])
print("First five discounted payoffs:", x[:5])


### 3. Estimate price and sampling uncertainty
For independent finite-variance discounted payoffs, estimate the standard error by sample standard deviation divided by the square root of N. The interval below is a large-sample approximation.


In [ ]:
price = x.mean()
se = x.std(ddof=1) / np.sqrt(N)
ci = (price - 1.96 * se, price + 1.96 * se)
print(f"Price {price:.6f}, SE {se:.6f}")
print(f"Approximate 95% CI [{ci[0]:.6f}, {ci[1]:.6f}]")


### 4. Compare with Black–Scholes
This reference uses the same no-dividend model and inputs. Agreement does not show that the model matches real markets. Handle zero time and zero volatility explicitly.


In [ ]:
def black_scholes_call(s0, k, r, sigma, t):
    if t == 0:
        return max(s0 - k, 0.)
    if sigma == 0:
        return max(s0 - k * math.exp(-r*t), 0.)
    phi = lambda a: .5 * (1 + math.erf(a / math.sqrt(2)))
    d1 = (math.log(s0/k) + (r + .5*sigma**2)*t) / (sigma*math.sqrt(t))
    d2 = d1 - sigma*math.sqrt(t)
    return s0*phi(d1) - k*math.exp(-r*t)*phi(d2)

benchmark = black_scholes_call(S0, K, r, sigma, T)
print(f"Black-Scholes {benchmark:.6f}")
print(f"Monte Carlo minus benchmark {price-benchmark:.6f}")


### 5. Investigate convergence
Use nested prefixes of one random stream. Estimates at different N are correlated. They need not move monotonically toward the benchmark. SE typically decreases at the square-root rate, so halving it takes about four times as many samples.


In [ ]:
z_all = np.random.default_rng(seed).standard_normal(1_000_000)
st_all = S0 * np.exp((r-.5*sigma**2)*T + sigma*np.sqrt(T)*z_all)
x_all = np.exp(-r*T) * np.maximum(st_all-K, 0.)
print("N          estimate       SE         approx 95% CI")
for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    v = x_all[:n]
    estimate = v.mean()
    err = v.std(ddof=1) / np.sqrt(n)
    print(f"{n:>9,}  {estimate:>11.6f}  {err:>9.6f}  [{estimate-1.96*err:.6f}, {estimate+1.96*err:.6f}]")


### 6. Easy checks
Use deterministic limiting cases rather than expecting every stochastic 95% interval to contain the benchmark. The full script also validates inputs and checks reproducibility and strike monotonicity.


In [ ]:
assert math.isclose(black_scholes_call(100,100,.05,.2,1), 10.450583572185565, abs_tol=1e-12)
assert black_scholes_call(110,100,.05,.2,0) == 10
deterministic_price = max(S0-K*np.exp(-r*T), 0.)
assert math.isclose(black_scholes_call(S0,K,r,0,T), deterministic_price)
assert np.all(x >= 0)
assert np.all(np.maximum(st-110,0) <= np.maximum(st-100,0))
print("Checks passed")


### 7. Optional: antithetic pairs
Use Z and minus Z. At the same 100,000-payoff budget, there are 50,000 independent pair averages. Compute SE using the pair averages, not the dependent individual payoffs.


In [ ]:
z_pair = np.random.default_rng(seed).standard_normal(50_000)
st_plus = S0*np.exp((r-.5*sigma**2)*T + sigma*np.sqrt(T)*z_pair)
st_minus = S0*np.exp((r-.5*sigma**2)*T - sigma*np.sqrt(T)*z_pair)
pairs = np.exp(-r*T)*(np.maximum(st_plus-K,0)+np.maximum(st_minus-K,0))/2
pair_se = pairs.std(ddof=1)/np.sqrt(len(pairs))
print(f"Antithetic price {pairs.mean():.6f}, SE {pair_se:.6f}")


### Exercises
1. Change N and then the seed. Explain which outputs change.
2. Set sigma to zero and predict the result first.
3. Increase K using common random draws. Why can the payoffs only decrease or stay equal?
4. Remove discounting. Why does a larger sample fail to fix the error?
5. Explain why a narrow confidence interval does not prove the market model is correct.

Across repeated runs, approximately 95% of intervals from this large-sample procedure would cover the fixed model price. A particular interval is not a guarantee. It excludes model and calibration error.

### References
- [NumPy standard-normal draws](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.standard_normal.html)
- [MIT OCW risk-neutral pricing and Black–Scholes](https://ocw.mit.edu/courses/18-s096-topics-in-mathematics-with-applications-in-finance-fall-2013/resources/mit18_s096f13_lecnote19/)
- [Repository and full reference script](https://github.com/manupriya-gupta-giga/monte-carlo-starter)


## Further audience challenges
Try an arithmetic-average Asian call and a floating-rate note with coupon memory.
[Challenge specifications](https://github.com/manupriya-gupta-giga/monte-carlo-starter/blob/main/CHALLENGES.md)
[Runnable rate reference](https://github.com/manupriya-gupta-giga/monte-carlo-starter/blob/main/rate_challenges.py)
The reference uses exact Vasicek rate transitions but approximate integrated rates. Plain floaters are discussed separately from the path-dependent extension.